In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from  pyspark.sql import functions as F, Window
from pyspark.sql.functions import *


In [ ]:
df_managment=spark.read.table('bronze_management')
df_product_spec = spark.read.table('bronze_product_spec')
df_calendar = spark.read.table('bronze_calendar')
df_car_stream = spark.read.table('bronze_CarSupplyChainGermanCar')

# Standardize column names 

In [ ]:
start_date_str = "2023-08-01"

w = Window.orderBy(F.monotonically_increasing_id())
silver_base = (
    df_car_stream
    .withColumnRenamed("fuel_efficiency_l_100", "fuel_efficiency_l_100km")
    .withColumnRenamed("weight","weight_kg")
    .withColumnRenamed("c02_emission_g_km","co2_emission_g_km")
    .withColumn("country",F.lit("DE"))
    .withColumn("order_date", df_car_stream["order_date"].cast("date"))
    .withColumn("expected_delivery", F.to_timestamp("expected_delivery"))
    .withColumn("rn" ,F.row_number().over(w))
    .withColumn("order_date",
        F.expr(
                        "date_add(date'{start}', (rn - 1) % (datediff(current_date(), date'{start}') + 1))"
            .format(start = start_date_str)
        )
    )
    .drop("rn")
)
              

In [ ]:
display(silver_base)

In [ ]:
df_car_stream.select("expected_delivery").distinct().show(10, False)


In [ ]:
display(silver_base)

# Split the Car_Supply_German_Car to Six tables : Vehicle, Manufacturing, Supply_chain, Process, Quality, Fact_events


****

In [ ]:
Silver_Vehicle = (
    silver_base
    .select(
        "car_brand",
        "car_model",
        "body_style",
        "color",
        "model_year",
        "engine_type",
        "fuel_type",
        "horsepower",
        "fuel_efficiency_l_100km",
        "weight_kg",
        "co2_emission_g_km"
        )
        
        .withColumn("vehicle_key", F.monotonically_increasing_id())
        )

display(Silver_Vehicle)



In [ ]:
silver_manufacturing = (
    silver_base
    .select(
        "plant_id",
        "assembly_line",
        "shift"
    )
    
    .withColumn("manufacturing_key",F.monotonically_increasing_id())
)
display(silver_manufacturing)

In [ ]:
silver_supply_chain = (
    silver_base
    .select(
        "part_id",
        "supplier_id",
        "warehouse",
        "transport_mode",
        "route"
    )
    
    .withColumn("supply_chain_key", F.monotonically_increasing_id())
)

display(silver_supply_chain)



In [ ]:
silver_process = (
    silver_base
    .select(
        "status",
        "order_date",
        "lead_time_days",
        "expected_delivery"
    )
    
    .withColumn("process_key", F.monotonically_increasing_id())
)

display(silver_process)

In [ ]:
silver_quality = (
    silver_base
    .select(
        "quality_score",
        "defect_flag"
        )
        
        .withColumn("quality_key", F.monotonically_increasing_id())
)

display(silver_quality)

In [ ]:
v = Silver_Vehicle.select(
    "car_brand","car_model","model_year","body_style", "color","engine_type","fuel_type",
        "horsepower", "fuel_efficiency_l_100km","weight_kg","co2_emission_g_km","vehicle_key"
)

m = silver_manufacturing.select(
    "manufacturing_key",
    "plant_id","assembly_line","shift"
)
s = silver_supply_chain.select(
    "supply_chain_key",
    "part_id","supplier_id","warehouse","transport_mode","route"
)
p = silver_process.select(
    "process_key",
    "status","order_date","lead_time_days","expected_delivery"
)
q= silver_quality.select(
    "quality_key",
    "quality_score","defect_flag"

)

silver_fact_event = (
    silver_base
    .join(v,["car_brand","car_model","model_year","body_style","color","engine_type",
    "fuel_type","horsepower","fuel_efficiency_l_100km","weight_kg","co2_emission_g_km"], "left")
    .join(m,["plant_id","assembly_line","shift"], "left")
    .join(s,["part_id","supplier_id","warehouse","transport_mode","route"],"left")
    .join(p,["status","order_date","lead_time_days","expected_delivery"],"left")
    .join(q,["quality_score","defect_flag"],"left")
    .select(
        "event_id",
        "timestamp",
        "vehicle_key",
        "manufacturing_key",
        "supply_chain_key",
        "process_key",
        "quality_key",
        "quantity",
        "stock_level",
        "country"
    )
)

display(silver_fact_event)

In [ ]:
silver_fact_event.select(
    "vehicle_key",
    "manufacturing_key",
    "supply_chain_key",
    "process_key",
    "quality_key"
).distinct().count()


# Build CLEAN dimension

# Correcting the SupplierAdress Column

In [ ]:
df_managment = df_managment.withColumn("SupplierAddress", \
regexp_replace(col("SupplierAddress"),r"^(\d+)\s+(.*)$","$2 $1"))

display(df_managment)

# Correcting the OrderDate& ShipDate

In [ ]:
df_managment = df_managment.withColumn("OrderDate", F.to_date(F.col("OrderDate"), "yyyy/MM/dd")) \
       .withColumn("ShipDate", F.to_date(F.col("ShipDate"), "yyyy/MM/dd"))

# 3. Determine the Target Year based on SupplierID
# 1-333   -> 2024
# 334-666 -> 2025
# 667-1000 -> 2026
df_managment = df_managment.withColumn("target_year", 
    F.when(F.col("SupplierID") <= 333, 2024)
     .when(F.col("SupplierID") <= 666, 2025)
     .otherwise(2026)
)
df_managment = df_managment.withColumn("year_diff", F.col("target_year") - F.year("OrderDate"))
#Shift OrderDate and ShipDate by the calculated year difference
# We use make_interval(years, months, weeks, days, hours, mins, secs)
df_managment = df_managment.withColumn("OrderDate", F.expr("OrderDate + make_interval(year_diff, 0, 0, 0, 0, 0, 0)")) \
             .withColumn("ShipDate", F.expr("ShipDate + make_interval(year_diff, 0, 0, 0, 0, 0, 0)"))
# 6. Drop the temporary calculation columns
df_managment = df_managment.drop( "target_year", "year_diff")
display(df_managment)



# Split Management Table to Dim_Customer, Dim_Supplier, Dim_product,Fact_sales

In [ ]:
# Dim_Customer
dim_customer = df_managment.select(
    "CustomerID", "CustomerName", "Gender", "JobTitle", 
    "PhoneNumber", "EmailAddress", "City", "Country", "State", "CustomerAddress"
).dropDuplicates(["CustomerID"])

# Dim_Supplier
dim_supplier = df_managment.select(
    "SupplierID", "SupplierName", "SupplierAddress", "SupplierContactDetails"
).dropDuplicates(["SupplierID"])

# Dim_Product
dim_product = df_managment.select("ProductID", "CarMaker", "CarModel", "CarColor", "CarModelYear") \
# Fact_Sales
fact_sales = df_managment.select(
    "OrderID", "OrderDate", "ShipDate", "SupplierID", "ProductID", "CustomerID",
    "Sales", "Quantity", "Discount", "ShipMode", "Shipping"
)



In [ ]:
display(fact_sales)

# Save tables in Silver Lake house 

In [ ]:
dim_customer.write.mode("overwrite").format('delta').save("abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_Dim_Customer")
dim_supplier.write.mode("overwrite").format('delta').save("abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_Dim_Supplier")
dim_product.write.mode("overwrite").format('delta').save("abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_Dim_Product")
fact_sales.write.mode("overwrite").format('delta').save("abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_Fact_Sales")
df_calendar.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_calendar')
df_product_spec.write.mode('overwrite').format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_product_spec')


In [ ]:
Silver_Vehicle.write.mode('overwrite').option("mergeSchema", "true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_vehicle')

In [ ]:
silver_manufacturing.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_manufacturing')

In [ ]:
silver_supply_chain.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_supply_chain')

In [ ]:
silver_process.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_process')

In [ ]:
silver_quality.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_quality')

In [ ]:
silver_fact_event.write.mode('overwrite').option("mergeSchema","true").format('delta').save('abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_fact_event')

In [ ]:
dim_customer.write.format("delta").mode("overwrite").save("abfss://cardataset@onelake.dfs.fabric.microsoft.com/Silver_LH.Lakehouse/Tables/dbo/silver_Dim_Customer")
